# CrimsonVC Cover Lite

A compact Google Colab workflow for making an **RVC AI Cover** without mounting Google Drive or opening the training interface.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DDME36/CrimsonVC-Studio/blob/main/CrimsonVC_RVC_Only.ipynb)

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Select **Runtime > Run all**.
3. Open the `gradio.live` URL from Cell 2.
4. Add a voice model under **Voice Models**, then return to **AI Cover**.

> Files are stored only in the temporary Colab runtime. Download the finished cover before deleting the runtime. Use `CrimsonVC_Colab.ipynb` when Google Drive persistence, speech conversion, training, or model benchmarks are needed.


In [ ]:
# @title 1. Install CrimsonVC Cover Lite
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

repository_url = "https://github.com/DDME36/CrimsonVC-Studio.git"  # @param {type:"string"}
repository_revision = "main"  # @param {type:"string"}
reset_workspace = False  # @param {type:"boolean"}
require_gpu = True  # @param {type:"boolean"}
python_version = "3.12"  # @param ["3.12"]
uv_version = "0.9.11"  # @param {type:"string"}

WORKSPACE = Path("/content/CrimsonVC")
DATA_ROOT = Path("/content/CrimsonVC-Lite-data")
TEMP_ROOT = Path("/content/CrimsonVC-Lite-temp")


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    """Run a command, stream all output, and stop when it fails."""
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Unable to capture subprocess output.")
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        print(f"\n[ERROR] Command failed with exit code {return_code}.", flush=True)
        raise SystemExit(return_code)


if not repository_url.startswith("https://github.com/"):
    raise ValueError("repository_url must be a GitHub HTTPS repository URL.")
if require_gpu and shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU is attached. Select Runtime > Change runtime type > T4 GPU, then retry."
    )
if shutil.which("nvidia-smi"):
    run([
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ])

if reset_workspace and WORKSPACE.exists():
    if WORKSPACE.resolve() != Path("/content/CrimsonVC"):
        raise RuntimeError(f"Refusing to remove unexpected path: {WORKSPACE}")
    shutil.rmtree(WORKSPACE)

if not WORKSPACE.exists():
    WORKSPACE.mkdir(parents=True)
    run(["git", "init"], cwd=WORKSPACE)
    run(["git", "remote", "add", "origin", repository_url], cwd=WORKSPACE)
elif not (WORKSPACE / "pyproject.toml").is_file():
    raise RuntimeError(
        f"{WORKSPACE} exists but is not a CrimsonVC checkout. Enable reset_workspace and retry."
    )

run(["git", "fetch", "--depth", "1", "origin", repository_revision], cwd=WORKSPACE)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=WORKSPACE)
os.chdir(WORKSPACE)
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=WORKSPACE, text=True
).strip()
print(f"Repository ready at commit {commit}: {WORKSPACE}")

models_dir = DATA_ROOT / "models"
audio_dir = DATA_ROOT / "audio"
config_dir = DATA_ROOT / "config"
for directory in (models_dir, audio_dir, config_dir, TEMP_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "URVC_MODELS_DIR": str(models_dir),
    "URVC_AUDIO_DIR": str(audio_dir),
    "URVC_CONFIG_DIR": str(config_dir),
    "URVC_TEMP_DIR": str(TEMP_ROOT),
    "URVC_CONSOLE_LOG_LEVEL": "WARNING",
    "URVC_DOWNLOAD_ALL_EMBEDDERS": "0",
    "GRADIO_ANALYTICS_ENABLED": "False",
    "PYTHONIOENCODING": "utf-8",
    "PYTHONUTF8": "1",
    "UV_CACHE_DIR": "/content/uv-cache",
    "UV_PROJECT_ENVIRONMENT": str(WORKSPACE / ".venv"),
})

run([sys.executable, "-m", "pip", "install", "--quiet", f"uv=={uv_version}"])
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv was installed but is not available on PATH.")
run([uv, "python", "install", python_version])
sync_command = [
    uv,
    "sync",
    "--python",
    python_version,
    "--extra",
    "cuda",
    "--no-dev",
    "--no-editable",
]
if (WORKSPACE / "uv.lock").is_file():
    sync_command.append("--locked")
run(sync_command, cwd=WORKSPACE)

VENV_PYTHON = WORKSPACE / ".venv" / "bin" / "python"
if not VENV_PYTHON.is_file():
    raise RuntimeError(f"Project Python was not found: {VENV_PYTHON}")
runtime_env = os.environ.copy()
runtime_env["PYTHONPATH"] = str(WORKSPACE / "src")
runtime_env["PYTHONUNBUFFERED"] = "1"
runtime_env["MPLBACKEND"] = "Agg"
run(
    [str(VENV_PYTHON), "-u", "-m", "ultimate_rvc.core.main"],
    cwd=WORKSPACE,
    env=runtime_env,
)

verification_code = r'''
import json
import gradio
import torch
report = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gradio": gradio.__version__,
}
print(json.dumps(report, indent=2))
if not torch.cuda.is_available():
    raise SystemExit("The project environment cannot access CUDA.")
'''
run([str(VENV_PYTHON), "-c", verification_code], cwd=WORKSPACE, env=runtime_env)
print("Cover Lite is installed. Cell 2 can now launch the Web UI.")


In [ ]:
# @title 2. Launch AI Cover Web UI
from getpass import getpass

enable_authentication = False  # @param {type:"boolean"}
auth_username = "crimsonvc"  # @param {type:"string"}
auth_password = ""  # @param {type:"string"}

if "WORKSPACE" not in globals() or "VENV_PYTHON" not in globals():
    raise RuntimeError("Run Cell 1 before launching Cover Lite.")

launch_env = os.environ.copy()
if enable_authentication:
    password = auth_password.strip() or getpass(
        "Create a Web UI password (minimum 8 characters): "
    ).strip()
    if len(password) < 8:
        raise ValueError("The Web UI password must contain at least 8 characters.")
    launch_env["URVC_AUTH_USERNAME"] = auth_username.strip() or "crimsonvc"
    launch_env["URVC_AUTH_PASSWORD"] = password
    print(f"Authentication enabled for user: {launch_env['URVC_AUTH_USERNAME']}")
else:
    launch_env.pop("URVC_AUTH_USERNAME", None)
    launch_env.pop("URVC_AUTH_PASSWORD", None)
    print("Authentication is disabled; the Gradio share URL opens directly.")

source_dir = WORKSPACE / "src"
existing_pythonpath = launch_env.get("PYTHONPATH")
launch_env["PYTHONPATH"] = (
    f"{source_dir}{os.pathsep}{existing_pythonpath}"
    if existing_pythonpath
    else str(source_dir)
)
launch_env["PYTHONUNBUFFERED"] = "1"
launch_env["MPLBACKEND"] = "Agg"
launch_env["URVC_UI_MODE"] = "cover"

print(f"Launching Cover Lite from: {source_dir}")
print("Stop this cell to shut down the public URL.")
run(
    [str(VENV_PYTHON), "-u", "-m", "ultimate_rvc.web.colab"],
    cwd=WORKSPACE,
    env=launch_env,
)
